In [6]:
# !pip3 install firebase-admin

In [1]:
import firebase_admin
from firebase_admin import credentials, firestore

/root/Medical-Symptoms-Checker-Bot/bot_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# Path to your downloaded Firebase private key JSON file
cred = credentials.Certificate("/root/Medical-Symptoms-Checker-Bot/health-app-7a8b0-firebase-adminsdk-fbsvc-9d28c0ae3f.json")

# Initialize Firebase App (only once per script)
firebase_admin.initialize_app(cred)

# Initialize Firestore client
db = firestore.client()

# Get all top-level collections
collections = db.collections()

print("Collections in Firestore:")
for collection in collections:
    print(collection.id)

Collections in Firestore:
admins
chatHistory
doctors
emergencyContacts
insurance
medicalReports
medicines
personalContacts
personalDetails
pharmacies
userPreferences
users


In [52]:
def get_user_medical_history_and_medicine_summary(uid: str):
    # ---------------------------
    # 1. Get medical history
    # ---------------------------
    user_ref = db.collection("users").document(uid)
    doc = user_ref.get()

    if not doc.exists:
        return {"error": "User not found"}

    data = doc.to_dict()
    name=data.get("displayName")
    
    summaries = []
    if "answers" in data:
        for item in data["answers"]:
            sum_ans = item.get("summarizedAnswer")
            if sum_ans:
                summaries.append(sum_ans)

    # ---------------------------
    # 3. Get chat history
    # ---------------------------
    ref = db.collection("chatHistory")
    query = ref.where("userId", "==", uid)

    messages = []

    for doc in query.stream():
        data = doc.to_dict()
        # print(data)
        sender = "user" if data.get("isUser") else "bot"

        messages.append({
            "message": data.get("message"),
            "sender": sender
        })


    # ---------------------------
    # 4. Build final response
    # ---------------------------
    return {
        "user_id": uid,
        'user_name':name,
        "medical_history": summaries,
        "chat_history":messages
    }


# Call function
result = get_user_medical_history_and_medicine_summary("pGSZNP7t8odnxmOIpRbDcuthbbM2")

print(result)

# print(result["user_id"])
# print("\n")
# print(result["medical_history"])
# print("\n")
# print(result["medicines"])

{'user_id': 'pGSZNP7t8odnxmOIpRbDcuthbbM2', 'user_name': 'Scott Grody', 'medical_history': ["Patient's date of birth is 07/06/1946.", 'Patient identifies as Male.', "Patient's ethnicity is Caucasian .", "Patient's address is 21377 Gosier Way.", 'Patient resides in Boca Raton.', 'Patient resides in Florida .', "Patient's zip code is 33428.", "Patient's phone number is 5617025533.", 'Patient\'s height is 5\'10".', "Patient's weight is 175 lbs.", 'Patient has had surgeries in the past.', 'Laser on left vocal cord ', 'Patient has been hospitalized in the past.', '2012 for vocal cord', 'Patient has high blood pressure.', 'Patient provided details about blood pressure.', 'Patient has diabetes.', 'Patient provided details about diabetes.', 'Patient does not have heart disease.', 'Patient does not have any allergies.', 'Patient currently smokes tobacco.', '1/2 pack a day', 'Patient consumes alcohol.', 'Socially ', 'Patient does not use recreational drugs.', 'Patient has not experienced recent 

In [13]:
# -----------------------
# Get all chatHistory documents
# -----------------------
def get_user_messages_clean(user_id: str,session_id: str):
    ref = db.collection("chatHistory")
    query = ref.where("userId", "==", user_id).where("sessionId", "==", session_id)
    
    all_data = []
    for doc in query.stream():
        data = doc.to_dict()
        all_data.append(data)
    
    sorted_data = sorted(all_data, key=lambda x: x['timestamp'])
    # print(sorted_data)    

    
    messages = []
    for data in sorted_data:
        role = "user" if data.get("isUser") else "assistant"
        messages.append({
            "role": role,
            "content": data.get("message", ""),
        })
    
    return {"messages": messages}
    

chats = get_user_messages_clean("Jfnpq4qbJKbhX9zcIOotI8OW20P2","1762126615626")
print(chats)

{'messages': [{'role': 'user', 'content': 'My wife of 30 years gives me a headache what should I do'}, {'role': 'assistant', 'content': 'When did this start, and how severe is it?'}, {'role': 'user', 'content': '28 years ago and very stressful'}, {'role': 'assistant', 'content': "It sounds like you're experiencing long-term stress related to your relationship, which can indeed lead to headaches. Stress management techniques such as regular exercise, mindfulness, and open communication with your spouse may help alleviate some of this tension. Additionally, consider seeking support from a counselor or therapist who can provide strategies for managing relationship stress. \n\nThese answers are evidence-based, derived from millions of medical data points, and tailored using your profile and medications when available. They are not a substitute for a doctor’s diagnosis or treatment plan.\n\nPossible Causes\n- Several common causes may fit based on your description.\n\nWhat You Can Do Now\n-

/tmp/ipykernel_902288/3729038428.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  query = ref.where("userId", "==", user_id).where("sessionId", "==", session_id)


In [46]:
def get_recent_chat_history(uid: str, limit: int = 10):
    """Get recent chat history - last N messages only"""
    try:
        ref = db.collection("chatHistory")
        query = ref.where("userId", "==", uid).order_by(
            "timestamp", direction=firestore.Query.DESCENDING
        ).limit(limit)
        
        messages = []
        for doc in query.stream():
            data = doc.to_dict()
            sender = "user" if data.get("isUser") else "bot"
            messages.append({
                "message": data.get("message", ""),
                "sender": sender,
                "timestamp": data.get("timestamp")
            })
        
        # Reverse to get chronological order
        messages.reverse()
        return messages
    except Exception as e:
        # Silently fail - chat history is optional
        return []


messages=get_recent_chat_history("pGSZNP7t8odnxmOIpRbDcuthbbM2")
print(messages)

[]


In [51]:
# ref = db.collection("chatHistory")
# query = ref.where("userId", "==", "pGSZNP7t8odnxmOIpRbDcuthbbM2")
# messages = []
# for doc in query.stream():
#     data = doc.to_dict()
#     sender = "user" if data.get("isUser") else "bot"
#     messages.append({
#         "message": data.get("message", ""),
#         "sender": sender,
#         "timestamp": data.get("timestamp")
#     })

# # Reverse to get chronological order
# messages.reverse()
# print(messages)

In [ ]:
user_data = {
  "user_id": "pGSZNP7t8odnxmOIpRbDcuthbbM2",
  "user_name": "Scott Grody",
  "user_bio":[
      {"question":"what is your date of birth?",
       "answer":"Patient's date of birth is 07/06/1946."
       },
      {"question":"what is your gender",
       "answer":"Patient identifies as Male."
       },
      {"question":"what is your ethnicity",
       "answer":"Patient's ethnicity is Caucasian ."
       },
      {"question":"what is your home address",
       "answer":"Patient's address is 21377 Gosier Way."
       },
      {"question":"what is your city ",
       "answer":"Patient resides in Boca Raton."
       },
      {"question":"what is your state",
       "answer":"Patient resides in Florida ."
       },
      {"question":"what is your zip code",
       "answer":"Patient's zip code is 33428."
       },
      {"question":"what is your phone number",
       "answer":"Patient's phone number is 5617025533."
       },
      {"question":"what is your height",
       "answer":"Patient's height is 5'10\"."
       },
      {"question":"what is your weight",
       "answer":"Patient's weight is 175 lbs."
       }
  ],
  "medical_history": [
    {
      "question": "Have you had any surgeries in the past?",
      "answer": "No"
    },
    {
      "question": "Have you ever been hospitalized?",
      "answer": "Yes, I have been hospitalized previously."
    },
    {
      "question": "Do you have high blood pressure?",
      "answer": "No"
    },
    {
      "question": "Do you have diabetes?",
      "answer": "Yes, I have diabetes."
    },
    {
      "question": "Do you have heart disease?",
      "answer": "No"
    },
    {
      "question": "Do you have any known allergies?",
      "answer": "Yes, I have known allergies."
    },
    {
      "question": "Do you currently smoke tobacco?",
      "answer": "No"
    },
    {
      "question": "Do you consume alcohol?",
      "answer": "Yes, I consume alcohol."
    },
    {
      "question": "Do you use recreational drugs?",
      "answer": "No"
    },
    {
      "question": "Have you experienced any recent weight changes?",
      "answer": "Yes, I have experienced recent weight changes."
    },
    {
      "question": "Have you had a fever in the past month?",
      "answer": "No"
    },
    {
      "question": "Do you have a history of cancer?",
      "answer": "Yes, I have a history of cancer."
    },
    {
      "question": "Is there any family history of serious illness (e.g., cancer, heart disease)?",
      "answer": "No"
    }
  ]

}